# Extension 4: neural TSP solver (POMO) vs classical methods

This notebook reproduces Extension 4 of [tsp-exact-vs-approx](https://github.com/samiksha-prasad/tsp-exact-vs-approx).

It runs the pretrained **POMO** models (Kwon et al., *POMO: Policy Optimization with Multiple Optima for Reinforcement Learning*, NeurIPS 2020, [code](https://github.com/yd-kwon/POMO)) on:
1. the 15 TSPLIB instances with published optima, and
2. random instances from 20 to 500 cities, to test how models trained at one size generalise to others.

POMO's code and weights are downloaded from the authors' repository; they are not part of this project.

**Before running:** *Runtime → Change runtime type → T4 GPU* (free). It also runs on CPU, more slowly.

In [ ]:
# 1. Get this project and POMO
!git clone -q https://github.com/samiksha-prasad/tsp-exact-vs-approx.git
!git clone -q --depth 1 https://github.com/yd-kwon/POMO.git
%cd tsp-exact-vs-approx
!pip install -q networkx

In [ ]:
# 2. Check the hardware
import torch
print("PyTorch", torch.__version__)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none, running on CPU")

In [ ]:
# 3. Quick check: POMO on one TSPLIB instance
import sys
sys.path.insert(0, "neural")
from pomo_wrapper import PomoSolver
from tsplib import load_instance
from tsp_solvers import tour_length

solver = PomoSolver("/content/POMO", train_size=100)
points, dist, optimum = load_instance("kroA100")
tour, secs = solver.solve(points)
length = tour_length(tour, dist)
print(f"kroA100: POMO tour {length:.0f}, optimum {optimum}, gap {100 * (length - optimum) / optimum:.2f}%, {secs:.2f}s")

In [ ]:
# 4. Full experiment (a few minutes on GPU, about 10-15 on CPU)
!python neural/run_neural.py --pomo-root /content/POMO

In [ ]:
# 5. Results
from IPython.display import Image, display
display(Image("neural_tsplib_gap.png"))
display(Image("neural_generalisation.png"))

In [ ]:
# 6. (Optional) download the result files
from google.colab import files
for f in ["neural_tsplib_results.csv", "neural_random_results.csv",
          "neural_tsplib_gap.png", "neural_generalisation.png"]:
    files.download(f)